## 1. Настройка

In [ ]:
from pathlib import Path
import os
import sys

GPU_ID = int(os.environ.get("DOOM_PROBE_GPU", "1"))
SEED = 1337
MAX_ACTIONS = int(os.environ.get("DOOM_PROBE_MAX_ACTIONS", "150"))
MODEL = Path(
    "/extra_disk_1/byzov-timofey/checkpoints/Qwen--Qwen3.6-35B-A3B-FP8/"
    "61a5771f218894aaacf97551e24a25b866750fc2"
)

if "torch" in sys.modules:
    raise RuntimeError("Restart the kernel before selecting CUDA_VISIBLE_DEVICES")
if not (MODEL / "config.json").is_file():
    raise FileNotFoundError(MODEL)
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"

# Используем MiniSGL из этого репозитория при запуске Jupyter из любой его папки.
repo = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "python/minisgl").is_dir()), None)
if repo is None:
    raise FileNotFoundError("Запустите Jupyter из репозитория MiniSGL")
sys.path.insert(0, str(repo / "python"))


## 2. Действия и prompt

In [ ]:
import asyncio
import hashlib
import time

import cv2
import gymnasium as gym
import ipywidgets as widgets
import torch
import vizdoom
import vizdoom.gymnasium_wrapper  # регистрирует VizdoomDeadlyCorridor-v1
from IPython.display import display
from minisgl.llm import AsyncLLM
from transformers import GenerationConfig

torch.set_num_threads(4)
ENV_ID = "VizdoomDeadlyCorridor-v1"
TEMPERATURE = 0.7
ACTIONS = [
    ("wait", "do nothing while enemies attack.", ()),
    ("right", "turn your view right to aim; do not move sideways.", ("TURN_RIGHT",)),
    ("left", "turn your view left to aim; do not move sideways.", ("TURN_LEFT",)),
    ("backward", "walk backward away from the corridor exit.", ("MOVE_BACKWARD",)),
    ("forward", "walk forward toward the corridor exit and green armor.", ("MOVE_FORWARD",)),
    ("fire", "shoot straight at the center of the screen.", ("ATTACK",)),
    ("east", "strafe sideways to your right without turning.", ("MOVE_RIGHT",)),
    ("west", "strafe sideways to your left without turning.", ("MOVE_LEFT",)),
]
ACTION_NAMES = [name for name, _, _ in ACTIONS]

TASK = (
    "Reach and touch the green armor at the far end of the corridor. Prefer forward whenever "
    "the corridor ahead is open; do not spend many turns firing or waiting at the start. "
    "Fire at a living enemy centered above the gun only when it threatens your route; turn "
    "toward a close off-center threat. Use sideways movement to dodge or pass an obstacle. "
    "Use the current frame and the change from the previous frame to avoid a blocked route."
)
PROMPT = (
    "You control a first-person player in Doom. You receive exactly two images: the previous "
    "observation, then the current observation. At the start they are identical. "
    "Decide from these images alone.\n"
    + TASK + "\nControls (one action lasts 4 game frames):\n"
    + "\n".join(f"{name}: {description}" for name, description, _ in ACTIONS)
    + "\nReturn exactly one of: " + ", ".join(ACTION_NAMES) + ".\n"
    + "Return only the action, without explanation."
)


## 3. Probe loop

In [ ]:
screen = widgets.Image(format="png", layout=widgets.Layout(width="480px"))
status = widgets.HTML()
display(widgets.VBox([screen, status]))


def update_dashboard(frame, step, action, score, armor, decision_s=0.0):
    screen.value = cv2.imencode(".png", cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))[1].tobytes()
    status.value = (
        f"step={step}  action={action}  score={score:+.1f}  "
        f"armor={armor:.0f}  decision={decision_s:.2f}s"
    )


def messages(previous, current):
    return [
        {"role": "system", "content": PROMPT},
        {"role": "user", "content": [
            {"type": "text", "text": "Previous observation:"},
            {"type": "image", "image": previous},
            {"type": "text", "text": "Current observation:"},
            {"type": "image", "image": current},
            {"type": "text", "text": "Choose the next action."},
        ]},
    ]


async def run_probe():
    env = gym.make(ENV_ID, render_mode="rgb_array", frame_skip=4, episode_timeout=2100)
    llm = None
    try:
        buttons = [button.name for button in env.unwrapped.game.get_available_buttons()]
        actual = tuple(
            tuple(buttons[i] for i, pressed in enumerate(row) if pressed)
            for row in env.unwrapped.button_map
        )
        assert actual == tuple(buttons for _, _, buttons in ACTIONS), actual
        observation, _ = env.reset(seed=SEED)
        previous = current = observation["screen"].copy()
        update_dashboard(current, 0, "ready", 0.0, 0.0)

        # Single-episode FP8/CUDA Graph profile; evaluation used N15.
        llm = AsyncLLM(
            str(MODEL),
            dtype=torch.bfloat16,
            quantization="fp8",
            max_running_req=4,
            memory_ratio=0.9,
            page_size=16,
            num_page_override=4096,
            max_seq_len_override=2048,
            attention_backend="fi",
            max_prefill_rows=512,
            cuda_graph_max_bs=0,  # простой запуск без CUDA Graph
            generation_config=GenerationConfig(
                do_sample=True, temperature=0.8, top_k=20, top_p=0.9,
            ),
        )
        # На этой машине FlashInfer импортируется после создания CUDA engine.
        import flashinfer.sampling as sampling

        encoded = [llm.tokenizer.encode(name, add_special_tokens=False) for name in ACTION_NAMES]
        assert all(len(ids) == 1 for ids in encoded)
        token_ids = [ids[0] for ids in encoded]
        assert len(set(token_ids)) == len(token_ids)
        agent_seed = int.from_bytes(
            hashlib.blake2s(f"{SEED}:agent".encode(), digest_size=4).digest(), "big"
        )
        generator = torch.Generator(device=llm.engine.device).manual_seed(agent_seed)
        game = env.unwrapped.game
        trace = []

        for step in range(MAX_ACTIONS):
            started = time.perf_counter()
            inputs = llm.processor.apply_chat_template(
                messages(previous, current),
                add_generation_prompt=True,
                enable_thinking=False,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
            )
            block = await llm.create_block()  # новый KV-блок на каждом шаге
            try:
                output = await llm(**inputs, cache_view=[block], write_to=block)
                logits = output.logits.reshape(-1)[token_ids].float()
                probabilities = torch.softmax(logits / TEMPERATURE, dim=0)
                selected = int(sampling.top_k_top_p_sampling_from_probs(
                    probabilities[None, :], len(token_ids), 1.0, generator=generator,
                ).item())
                probabilities = probabilities.tolist()
            finally:
                await llm.free_block(block)

            decision_s = time.perf_counter() - started
            observation, reward, terminated, truncated, _ = env.step(selected)
            trace.append({
                "step": step + 1,
                "action": ACTION_NAMES[selected],
                "probabilities": dict(zip(ACTION_NAMES, probabilities)),
                "reward": float(reward),
                "score": float(game.get_total_reward()),
                "armor": float(game.get_game_variable(vizdoom.GameVariable.ARMOR)),
                "decision_s": decision_s,
            })
            update_dashboard(
                observation["screen"] if not (terminated or truncated) else current,
                step + 1, ACTION_NAMES[selected], trace[-1]["score"],
                trace[-1]["armor"], decision_s,
            )
            if terminated or truncated:
                break
            previous, current = current, observation["screen"].copy()

        armor = float(game.get_game_variable(vizdoom.GameVariable.ARMOR))
        result = {
            "seed": SEED,
            "actions": len(trace),
            "final_score": float(game.get_total_reward()),
            "success": armor > 0 and game.is_episode_finished() and not game.is_player_dead(),
            "dead": bool(game.is_player_dead()),
        }
        return trace, result
    finally:
        env.close()
        if llm is not None:
            await llm.close()


## 4. Запуск и короткий trace

In [ ]:
trace, result = await run_probe()
print(result)
print("step  action     p(action)   reward     score   armor")
for row in trace[:12]:
    probability = row["probabilities"][row["action"]]
    print(f"{row['step']:>4}  {row['action']:<9}  {probability:.3f}"
          f"     {row['reward']:>7.3f}  {row['score']:>8.3f}  {row['armor']:>5.0f}")
print("first-step probabilities:", trace[0]["probabilities"])
